## Data Validation

Before any modeling we freeze the dataset and confirm it is large and dense enough.
All numbers below are produced by the code, not hard-coded here.

### Gate

- Users ≥ 1000
- Games ≥ 1000
- Interactions ≥ 100k

If the gate fails, change the sampling now — not after training.

### Decision

The sample is frozen at this stage and will not change afterwards.
The evaluation split is leave-last-out per user (the most recent interaction of each user goes to test).

## Проверка данных

Перед моделированием фиксируем датасет и убеждаемся, что он достаточно большой и плотный.
Все числа ниже выводятся кодом, а не захардкожены здесь.

### Гейт

- Пользователей ≥ 1000
- Игр ≥ 1000
- Взаимодействий ≥ 100k

Если гейт не пройден — меняем сэмплирование сейчас, а не после обучения.

### Решение

Выборка фиксируется на этом этапе и дальше не меняется.
Сплит для оценки — leave-last-out per user (последнее взаимодействие каждого пользователя уходит в test).

In [1]:
import pandas as pd
from pathlib import Path

# Project root directory
# Корневая директория проекта
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

# Load the frozen processed tables
# Загружаем зафиксированные обработанные таблицы
inter = pd.read_parquet(DATA / "processed" / "interactions.parquet")
games = pd.read_parquet(DATA / "processed" / "games.parquet")
print("loaded:", inter.shape, games.shape)

loaded: (901137, 5) (2759, 12)


In [2]:
# Core dataset statistics
# Основные статистики датасета
n_users = inter["user_id"].nunique()
n_games = inter["game_id"].nunique()
n_inter = len(inter)
sparsity = 1 - n_inter / (n_users * n_games)

print("users:       ", f"{n_users:,}")
print("games:       ", f"{n_games:,}")
print("interactions:", f"{n_inter:,}")
print("sparsity:    ", f"{sparsity:.4%}")
print("positive label share:", f"{inter['label'].mean():.3f}")

# Activity distribution (cold-start view)
# Распределение активности (взгляд на cold-start)
upc = inter["user_id"].value_counts()
gpc = inter["game_id"].value_counts()
print("\ninteractions per user  min/median/max:", upc.min(), int(upc.median()), upc.max())
print("interactions per game  min/median/max:", gpc.min(), int(gpc.median()), gpc.max())

users:        113,552
games:        2,872
interactions: 901,137
sparsity:     99.7237%
positive label share: 0.902

interactions per user  min/median/max: 5 6 222
interactions per game  min/median/max: 20 68 15975


In [3]:
# Validation gate: dataset must be large enough before modeling
# Гейт валидации: датасет должен быть достаточно большим до моделирования
assert n_users >= 1_000, "too few users"
assert n_games >= 1_000, "too few games"
assert n_inter >= 100_000, "too few interactions"
print("validation gate passed")

validation gate passed
